# Transform Constructors Data

1. Read bronze `constructors` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`)
1. Rename columns to make them more meaningful (`name` → `constructor_name`)
1. Remove duplicate records
1. Transform values of column `nationality` to Title Case
1. Write the transformed data to silver `constructors` table

In [0]:
%run "/Workspace/Users/deepanshu.patil69@gmail.com/Formula1/common/01_Environmnet_config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
const_df = spark.read.table(bronze_table)
display(const_df)

In [0]:
const_drop_url_df = const_df.drop("url")

In [0]:
display(
    const_drop_url_df.select([
        F.count(
            F.when(F.col(c).isNull(), c)).alias(c)
        for c in const_drop_url_df.columns
    ])
)

In [0]:
const_renamed_col_df = const_drop_url_df.withColumnsRenamed(
    {"constructorId": "constructor_id", "name": "constructor_name"}
)
display(const_renamed_col_df)

In [0]:
display(
    const_renamed_col_df
        .groupBy(const_renamed_col_df.columns)
        .agg(F.count(F.col('*')).alias("count"))
        .filter(F.col('count') > 1)
)

In [0]:
const_removed_duplicates = (
    const_renamed_col_df.dropDuplicates()
)
display(const_removed_duplicates)

In [0]:
const_final_df = (
    const_removed_duplicates
        .withColumn("nationality", F.initcap(F.col("nationality")))
)
const_final_df.display()

In [0]:
(
    const_final_df
        .write
        .format("delta")
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
select * from formula1.silver.constructors